In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

DenseIndex.embeddings:  (2107648, 1024)


In [18]:
import citation_utils
from collections import Counter
from dataclasses import dataclass, field
import random
from pathlib import Path
import json

# ---------------------------------------------------------------------------
# 配置
# ---------------------------------------------------------------------------
 
@dataclass
class SampleConfig:
    top_k: int = 100              # 每个 query 召回的 doc 数
    hard_neg_ratio: int = 6       # 每个正样本对应几个难负样本
    val_ratio: float = 0.1        # 验证集比例
    min_citations_in_doc: int = 1 # doc 至少含几个 citation 才纳入候选
    output_dir: str = "../ft_data"
    seed: int = 42
 
 
CFG = SampleConfig()
 
 
# ---------------------------------------------------------------------------
# 统计全局 citation 频次（用于软标签 quality_score）
# ---------------------------------------------------------------------------
 
def build_global_citation_freq(
    queries: list[str],
    ground_truths: dict[str, list[str]],
) -> dict[str, int]:
    """
    统计所有 ground truth citation 在 ground_truth 值中出现的总次数。
    这是对"全局引用频次"的近似——你也可以换成从整个语料库统计的真实频次。
    """
    counter: Counter = Counter()
    for citations in ground_truths.values():
        counter.update(citations)
    return dict(counter)


# ---------------------------------------------------------------------------
# 单个 query 的样本生成
# ---------------------------------------------------------------------------
 
def build_samples_for_query(
    query: str,
    gt_citations: list[str],       # 该 query 的 ground truth citation 列表
    global_freq: dict[str, int],
    cfg: SampleConfig,
) -> list[dict]:
    """
    召回 top_k 个 doc，按是否命中 ground truth citation 分为正/负样本。
    返回该 query 产生的所有样本列表。
    """
    gt_set = set(gt_citations)
    total_expected = len(gt_set)
 
    # 召回
    # hits 每个元素结构：{'citation': citation, 'text': text}
    hits = sparse_index.search_with_score(query, top_k=cfg.top_k)
 
    positives = []
    negatives = []
 
    for hit,_ in hits:
        doc_text = hit["text"]
 
        # 从 doc 中抽取所有 citation
        doc_citations = citation_utils.extract_citations_from_text(doc_text)
        if not doc_citations:
            continue  # 没有任何 citation 的 doc 跳过，对训练无帮助
 
        # 命中的 ground truth citation
        matched = [c for c in doc_citations if c in gt_set]
 
        sample = {
            "query": query,
            "doc": doc_text,
            "matched_citations": [
                {
                    "citation_id": c,
                    "global_freq": global_freq.get(c, 1),
                }
                for c in matched
            ],
            "total_expected": total_expected,
        }
 
        if matched:
            sample["label"] = 1.0
            positives.append(sample)
        else:
            sample["label"] = 0.0
            negatives.append(sample)
 
    if not positives:
        # 该 query 没有召回任何正样本，跳过（recall 阶段的问题，不在 rerank 训练范围内）
        print(f"query 无正样本，跳过：{query[:60]}")
        return []
 
    # 从负样本中取难负样本（BM25/稀疏召回的 top 负样本天然就是难负样本）
    n_neg = min(len(negatives), len(positives) * cfg.hard_neg_ratio)
    sampled_negatives = negatives[:n_neg]  # hits 已按相关性降序，前面的负样本更难
 
    return positives + sampled_negatives


# ---------------------------------------------------------------------------
# 主流程
# ---------------------------------------------------------------------------
 
def build_dataset(
    queries: list[str],
    ground_truths: dict[str, list[str]],  # {query: [citation1, citation2, ...]}
    cfg: SampleConfig = CFG,
):
    """
    遍历所有 query，生成训练样本，写入 train.jsonl 和 val.jsonl。
    """
    random.seed(cfg.seed)
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
 
    print(f"共 {len(queries)} 个 query，开始生成样本...")
 
    # 统计全局 citation 频次
    global_freq = build_global_citation_freq(queries, ground_truths)
    print(f"unique citation 数：{len(global_freq)}")
 
    all_samples = []
    skipped = 0
 
    for i, query in enumerate(queries):
        gt_citations = ground_truths.get(query, [])
        if not gt_citations:
            skipped += 1
            continue
 
        samples = build_samples_for_query(query, gt_citations, global_freq, cfg)
        all_samples.extend(samples)
 
        if (i + 1) % 100 == 0:
            print(f"  进度：{i+1}/{len(queries)}，当前样本数：{len(all_samples)}")
 
    print(f"生成完毕：{len(all_samples)} 条样本，{skipped} 个 query 因无 ground truth 跳过")
 
    # 统计正负比
    n_pos = sum(1 for s in all_samples if s["label"] == 1.0)
    n_neg = len(all_samples) - n_pos
    print(f"正样本：{n_pos}，负样本：{n_neg}，正负比 1:{n_neg//max(n_pos,1)}")
 
    # 打乱并切分
    random.shuffle(all_samples)
    n_val = int(len(all_samples) * cfg.val_ratio)
    val_samples   = all_samples[:n_val]
    train_samples = all_samples[n_val:]
 
    # 写文件
    for split, samples in [("train", train_samples), ("val", val_samples)]:
        path = f"{cfg.output_dir}/{split}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for s in samples:
                f.write(json.dumps(s, ensure_ascii=False) + "\n")

In [19]:
train_df = pd.read_csv("../data/train_rewrite_001.csv")
query_2_gold_citations_d = {}
train_id_2_query_d = {}
for query_id, gold_citations, query in zip(train_df['query_id'], train_df['gold_citations'], train_df['query2']):
    query_2_gold_citations_d[query] = gold_citations.split(';')
    train_id_2_query_d[query_id] = query

In [20]:
build_dataset([v for k,v in train_id_2_query_d.items()], query_2_gold_citations_d)

共 1139 个 query，开始生成样本...
unique citation 数：2695
query 无正样本，跳过：Römer-Statut §7 Abs.1 lit.g, i, k, §8 Abs.2 lit.a iv, c i – 
query 无正样本，跳过：Durchlässigkeitsgedanke Strafvollzug Rechtsprobleme
query 无正样本，跳过：Finanzmarktaufsicht Zuständigkeitsabgrenzung Sanktionen Stra
query 无正样本，跳过：Arbeitsrecht · Abmahnung · Verhaltensvorwurf · sexuelle Belä
query 无正样本，跳过：Anleiheemission, Obligationen, Publikumsofferte, Mindeststüc
query 无正样本，跳过：Beta Immobilien AG · Grundstück Y Kanton Zürich · Verkauf an
query 无正样本，跳过：Arbeitsschutzrecht; Überwachungssysteme am Arbeitsplatz; Vid
query 无正样本，跳过：Schiedsvereinbarung – Mediationsvorbehalt – Zuständigkeit de
query 无正样本，跳过：Schweiz, Schweizerisches Zivilprozessrecht, Begründungspflic
query 无正样本，跳过：Schweiz Immaterialgüterrechtsgesetz Kartellrecht Normenquell
query 无正样本，跳过：Handelsrechtliche Beurteilung: Veräußerung von Liegenschaft 
query 无正样本，跳过：Haftung für Personenschäden bei Besuch in Privatwohnung – Tr
query 无正样本，跳过：Schadensersatzanspruch bei Sachbeschädigung im 